# YOHO: Construccion tecnica de la imagen RGB de 3 canales

## Principio de diseno

Cada canal (R, G, B) es una **matriz 2D independiente** de valores en escala de grises (0-255). No es una imagen RGB. Las tres matrices se calculan por separado, se normalizan por separado, y solo al final se **apilan** en un array de forma `(alto, ancho, 3)` para formar la imagen RGB.

Eso garantiza que:
- Modificar el canal R no afecta lo que se ve en G o B.
- Cada canal contiene informacion unica de su representacion.
- La imagen RGB final es simplemente el empaquetado de las tres matrices, sin sumas ni mezclas.

## Rango de frecuencia comun: 20 Hz a Nyquist

Los tres canales cubren el mismo rango auditivo: **20 Hz a Nyquist (22050 Hz)**. Se excluye la banda de 0 a 20 Hz porque es ruido de baja frecuencia sin contenido informativo para las clases de DataSED. Esto tambien reduce el ruido en las imagenes del dataset.

## Tamano cuadrado

El tamano de la imagen final es `(N_FILAS, n_frames)`. Con `N_FILAS=1024` y una ventana de 24 s con `HOP_LENGTH=1024`, se generan ~1032 frames. La relacion de aspecto es ~0.99 (casi cuadrado).

In [ ]:
import sys
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image

sys.path.append("../src")
from config import AUDIO_DIR, CSV_MONO

plt.style.use('dark_background')

## 1. Parametros de analisis

Todos los parametros visibles en un solo bloque. Los tres canales cubren de 20 Hz a Nyquist.

In [ ]:
audio_name = "S-0219.wav"

In [ ]:
# Parametros de audio
SR = 44100
N_FFT = 2048
HOP_LENGTH = 1024

# Rango de frecuencias comun (20 Hz a Nyquist)
NYQUIST = SR / 2
FREQ_MIN = 20.0
FREQ_MAX = NYQUIST
N_FILAS = 1024

# CQT
CQT_BINS_POR_OCTAVA = 12
CQT_N_BINS = 120
CQT_FMIN = 20.0

# Parametros del canal R
R_VMIN = -70
R_VMAX = -6
R_INVERTIR_X = False
R_INVERTIR_Y = False
# Parametros del canal G
G_VMIN = -50
G_VMAX = -3
G_INVERTIR_X = False
G_INVERTIR_Y = False
# Parametros del canal B
B_VMIN = -50
B_VMAX = -6
B_INVERTIR_X = False
B_INVERTIR_Y = False

print(f"SR: {SR} Hz")
print(f"Nyquist: {NYQUIST} Hz")
print(f"FREQ_MIN: {FREQ_MIN} Hz")
print(f"FREQ_MAX: {FREQ_MAX} Hz")
print(f"N_FILAS: {N_FILAS}")

## 2. Cargar audio y calcular la base STFT

In [ ]:
audio_path = AUDIO_DIR / audio_name
y, sr = librosa.load(audio_path, sr=SR, mono=True)
duracion = len(y) / sr

stft = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)
mag_stft = np.abs(stft)
mag_stft_db = librosa.amplitude_to_db(mag_stft, ref=np.max)
freqs_stft = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)
times_stft = librosa.frames_to_time(np.arange(mag_stft.shape[1]), sr=sr, hop_length=HOP_LENGTH)

n_frames = mag_stft_db.shape[1]

print(f"Audio: {audio_name}")
print(f"Duracion: {duracion:.2f} s")
print(f"Frames STFT: {n_frames}")
print(f"Filas objetivo: {N_FILAS}")
print(f"Relacion de aspecto: {N_FILAS / n_frames:.3f}")

## 3. Canal R: STFT lineal en dB (20 Hz a Nyquist)

**Parametros de este bloque:**
- Rango de frecuencia: 20 Hz a Nyquist.
- Rejilla lineal de `N_FILAS` filas.
- `vmin`, `vmax`: -80 a 0 dB.
- Inversion: ninguna por defecto.

In [ ]:
# Rejilla lineal de 20 Hz a Nyquist
freqs_lineal = np.linspace(FREQ_MIN, FREQ_MAX, N_FILAS)

# Recortar STFT al rango 20 Hz - Nyquist
mask_r = (freqs_stft >= FREQ_MIN) & (freqs_stft <= FREQ_MAX)
freqs_r = freqs_stft[mask_r]
mag_r = mag_stft_db[mask_r, :]

# Interpolar a la rejilla lineal
canal_r = np.zeros((N_FILAS, n_frames), dtype=np.float32)
for i in range(n_frames):
    canal_r[:, i] = np.interp(freqs_lineal, freqs_r, mag_r[:, i])

canal_r_norm = np.clip((canal_r - R_VMIN) / (R_VMAX - R_VMIN), 0, 1)
canal_r_u8 = (canal_r_norm * 255).astype(np.uint8)

if R_INVERTIR_X:
    canal_r_u8 = np.fliplr(canal_r_u8)
if R_INVERTIR_Y:
    canal_r_u8 = np.flipud(canal_r_u8)

print(f"Canal R: rejilla {freqs_lineal[0]:.0f} Hz - {freqs_lineal[-1]:.0f} Hz")
print(f"Canal R (u8): forma={canal_r_u8.shape}, rango=[{canal_r_u8.min()}, {canal_r_u8.max()}]")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
img = ax.imshow(canal_r_u8, aspect='auto', origin='lower', cmap='Reds_r', vmin=0, vmax=255)

ticks_hz = [20, 100, 1000, 5000, 10000, 15000, 20000, 22050]
ticks_filas = [int(np.argmin(np.abs(freqs_lineal - hz))) for hz in ticks_hz]
ax.set_yticks(ticks_filas)
ax.set_yticklabels([f"{hz/1000:.1f}k" if hz >= 1000 else str(hz) for hz in ticks_hz])

ax.set_xlabel("Frames")
ax.set_ylabel("Frecuencia (Hz)")
ax.set_title(f"Canal R: STFT lineal (20 Hz - Nyquist) (vmin={R_VMIN}, vmax={R_VMAX})")
cbar = fig.colorbar(img, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Valor normalizado (0-255)")
plt.tight_layout()
plt.show()

## 4. Canal G: STFT log en dB (20 Hz a Nyquist)

**Parametros de este bloque:**
- Rango de frecuencia: 20 Hz a Nyquist.
- Rejilla logaritmica de `N_FILAS` filas.
- `vmin`, `vmax`: -80 a 0 dB.
- Inversion: ninguna por defecto.

In [ ]:
# Rejilla logaritmica de 20 Hz a Nyquist
freqs_log = np.logspace(np.log10(FREQ_MIN), np.log10(FREQ_MAX), N_FILAS)

# Interpolar a la rejilla log
canal_g = np.zeros((N_FILAS, n_frames), dtype=np.float32)
for i in range(n_frames):
    canal_g[:, i] = np.interp(freqs_log, freqs_r, mag_r[:, i])

canal_g_norm = np.clip((canal_g - G_VMIN) / (G_VMAX - G_VMIN), 0, 1)
canal_g_u8 = (canal_g_norm * 255).astype(np.uint8)

if G_INVERTIR_X:
    canal_g_u8 = np.fliplr(canal_g_u8)
if G_INVERTIR_Y:
    canal_g_u8 = np.flipud(canal_g_u8)

print(f"Canal G: rejilla {freqs_log[0]:.0f} Hz - {freqs_log[-1]:.0f} Hz")
print(f"Canal G (u8): forma={canal_g_u8.shape}, rango=[{canal_g_u8.min()}, {canal_g_u8.max()}]")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
img = ax.imshow(canal_g_u8, aspect='auto', origin='lower', cmap='Greens_r', vmin=0, vmax=255)

ticks_hz = [20, 100, 1000, 5000, 10000, 15000, 20000, 22050]
ticks_filas = [int(np.argmin(np.abs(freqs_log - hz))) for hz in ticks_hz]
ax.set_yticks(ticks_filas)
ax.set_yticklabels([f"{hz/1000:.1f}k" if hz >= 1000 else str(hz) for hz in ticks_hz])

ax.set_xlabel("Frames")
ax.set_ylabel("Frecuencia (Hz, log)")
ax.set_title(f"Canal G: STFT log (20 Hz - Nyquist) (vmin={G_VMIN}, vmax={G_VMAX})")
cbar = fig.colorbar(img, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Valor normalizado (0-255)")
plt.tight_layout()
plt.show()

## 5. Canal B: CQT (20 Hz a Nyquist)

**Parametros de este bloque:**
- `bins_per_octave`: 12.
- `n_bins`: 120.
- `fmin`: 20 Hz.
- `vmin`, `vmax`: -80 a 0 dB.
- Inversion: ninguna por defecto.

In [ ]:
# Calcular CQT
cqt = np.abs(librosa.cqt(
    y, sr=sr, hop_length=HOP_LENGTH,
    fmin=CQT_FMIN, n_bins=CQT_N_BINS, bins_per_octave=CQT_BINS_POR_OCTAVA
))
cqt_db = librosa.amplitude_to_db(cqt, ref=np.max)
freqs_cqt = librosa.cqt_frequencies(n_bins=CQT_N_BINS, fmin=CQT_FMIN, bins_per_octave=CQT_BINS_POR_OCTAVA)

# Interpolar a la rejilla log
canal_b = np.zeros((N_FILAS, n_frames), dtype=np.float32)
for i in range(n_frames):
    canal_b[:, i] = np.interp(freqs_log, freqs_cqt, cqt_db[:, i])

canal_b_norm = np.clip((canal_b - B_VMIN) / (B_VMAX - B_VMIN), 0, 1)
canal_b_u8 = (canal_b_norm * 255).astype(np.uint8)

if B_INVERTIR_X:
    canal_b_u8 = np.fliplr(canal_b_u8)
if B_INVERTIR_Y:
    canal_b_u8 = np.flipud(canal_b_u8)

print(f"Canal B: CQT desde {freqs_cqt[0]:.0f} Hz, interpolado a {freqs_log[0]:.0f}-{freqs_log[-1]:.0f} Hz")
print(f"Canal B (u8): forma={canal_b_u8.shape}, rango=[{canal_b_u8.min()}, {canal_b_u8.max()}]")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
img = ax.imshow(canal_b_u8, aspect='auto', origin='lower', cmap='Blues_r', vmin=0, vmax=255)

ticks_hz = [20, 100, 1000, 5000, 10000, 15000, 20000, 22050]
ticks_filas = [int(np.argmin(np.abs(freqs_log - hz))) for hz in ticks_hz]
ax.set_yticks(ticks_filas)
ax.set_yticklabels([f"{hz/1000:.1f}k" if hz >= 1000 else str(hz) for hz in ticks_hz])

ax.set_xlabel("Frames")
ax.set_ylabel("Frecuencia (Hz, log)")
ax.set_title(f"Canal B: CQT (20 Hz - Nyquist) (vmin={B_VMIN}, vmax={B_VMAX})")
cbar = fig.colorbar(img, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Valor normalizado (0-255)")
plt.tight_layout()
plt.show()

## 6. Verificacion: los tres canales son matrices 2D independientes

In [ ]:
print(f"canal_r_u8: shape={canal_r_u8.shape}, dtype={canal_r_u8.dtype}")
print(f"canal_g_u8: shape={canal_g_u8.shape}, dtype={canal_g_u8.dtype}")
print(f"canal_b_u8: shape={canal_b_u8.shape}, dtype={canal_b_u8.dtype}")

r_flat = canal_r_u8.flatten().astype(float)
g_flat = canal_g_u8.flatten().astype(float)
b_flat = canal_b_u8.flatten().astype(float)

print(f"\nCorrelacion R-G: {np.corrcoef(r_flat, g_flat)[0,1]:.4f}")
print(f"Correlacion R-B: {np.corrcoef(r_flat, b_flat)[0,1]:.4f}")
print(f"Correlacion G-B: {np.corrcoef(g_flat, b_flat)[0,1]:.4f}")

## 7. Apilar los tres canales en la imagen RGB final

Se crea un array de forma `(N_FILAS, n_frames, 3)` donde:
- `rgb[:, :, 0] = canal_r_u8` (rojo)
- `rgb[:, :, 1] = canal_g_u8` (verde)
- `rgb[:, :, 2] = canal_b_u8` (azul)

No hay suma ni mezcla.

In [ ]:
rgb_final = np.stack([canal_r_u8, canal_g_u8, canal_b_u8], axis=-1)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

axes[0].imshow(rgb_final, aspect='auto', origin='lower')
axes[0].set_title("Imagen RGB final")
axes[0].axis('off')

axes[1].imshow(canal_r_u8, aspect='auto', origin='lower', cmap='Reds_r', vmin=0, vmax=255)
axes[1].set_title("Componente R")
axes[1].axis('off')

axes[2].imshow(canal_g_u8, aspect='auto', origin='lower', cmap='Greens_r', vmin=0, vmax=255)
axes[2].set_title("Componente G")
axes[2].axis('off')

axes[3].imshow(canal_b_u8, aspect='auto', origin='lower', cmap='Blues_r', vmin=0, vmax=255)
axes[3].set_title("Componente B")
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 8. Imagen RGB final junto a su version con cajas

In [ ]:
# Cargar anotaciones y filtrar el audio de ejemplo
df_anot = pd.read_csv(CSV_MONO)
df_audio = df_anot[df_anot['sound_name'] == audio_name].copy()

# Paleta para las clases
_PALETA = ['#FF6B6B', '#4ECDC4', '#FFE66D', '#A78BFA', '#F472B6',
           '#34D399', '#FB923C', '#60A5FA', '#F87171', '#C084FC',
           '#2DD4BF', '#FBBF24', '#818CF8', '#FCA5A5', '#6EE7B7']

clases_unicas = df_audio['class_name'].unique().tolist()
color_por_clase = {c: _PALETA[i % len(_PALETA)] for i, c in enumerate(clases_unicas)}

print(f"Eventos en {audio_name}: {len(df_audio)}")
print(f"Clases: {clases_unicas}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

axes[0].imshow(rgb_final, aspect='auto', origin='lower')
axes[0].set_title(f"Imagen RGB final - {audio_name}")
axes[0].axis('off')

ax = axes[1]
ax.imshow(rgb_final, aspect='auto', origin='lower', extent=[0, duracion, 0, N_FILAS])

for _, row in df_audio.iterrows():
    t_ini = row['start_time']
    t_fin = row['end_time']
    clase = row['class_name']
    color = color_por_clase[clase]

    rect = patches.Rectangle(
        (t_ini, 0), t_fin - t_ini, N_FILAS,
        linewidth=2, edgecolor=color, facecolor=color, alpha=0.15
    )
    ax.add_patch(rect)
    ax.text(
        (t_ini + t_fin) / 2, N_FILAS * 0.05, clase,
        ha='center', va='bottom', fontsize=7, color='white',
        bbox=dict(boxstyle='round,pad=0.2', facecolor=color,
                  alpha=0.85, edgecolor='none')
    )

ax.set_xlim(0, duracion)
ax.set_xlabel("Tiempo (s)")
ax.set_title(f"Con cajas ground truth ({len(df_audio)} eventos)")
ax.set_yticks([])
ax.set_ylabel("")

plt.tight_layout()
plt.show()

## 9. Guardar la imagen RGB

In [ ]:
# Guardar la imagen con la orientación correcta
# (frecuencias bajas abajo, altas arriba)
rgb_para_guardar = np.flipud(rgb_final)
output_path = Path(f"RGB_final_{audio_name.replace('.wav', '.png')}")

img_pil = Image.fromarray(rgb_para_guardar, mode='RGB')
img_pil.save(output_path, format='PNG')
print(f"Guardada: {output_path}")
print(f"Modo: {img_pil.mode}, Tamano: {img_pil.size}")

## 10. Conclusiones

### Lo que se garantizo

1. **Todos los canales cubren 20 Hz a Nyquist** (22050 Hz), excluyendo la banda de 0-20 Hz que era ruido.
2. **Cada canal es una matriz 2D independiente** (uint8, forma `(N_FILAS, n_frames)`).
3. **La imagen RGB es solo el apilado** de las tres matrices. Verificado con `np.array_equal`.
4. **Tamano casi cuadrado**: `N_FILAS=1024` con ~1032 frames da relacion de aspecto ~0.99.
5. **Parametros visibles** en cada bloque: `vmin`, `vmax`, `invertir_x`, `invertir_y`.

### Rangos de frecuencia efectivos

| Canal | Rango efectivo | Rejilla |
|---|---|---|
| R | 20 Hz - 22050 Hz | Lineal (1024 filas) |
| G | 20 Hz - 22050 Hz | Logaritmica (1024 filas) |
| B | 20 Hz - 22050 Hz (CQT) | Logaritmica con Q constante, interpolada a 1024 filas |

### Siguientes pasos

1. Integrar esta logica en `dataset_builder.py`.
2. Ejecutar `split_dataset.py`.
3. Entrenar con YOLO26n.
4. Evaluar sobre test.
5. Comparar contra el baseline de 1 canal.

## Resultados


`(aexxx) aexxtool@AExxTool:~/YOHO-by-AEx/src$ python3 train.py
============================================================
ENTRENAMIENTO YOLO26n - DATASET RGB (3 CANALES)
============================================================
Modelo: yolo26n.pt
Data:   /mnt/f/DATASETS/DataSED/yolo_dataset/data.yaml
Existe: True
Epochs: 128
Batch:  -1
Imgsz:  1024
Device: 0
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo…: 100% ━━━━━━━━━━━━ 5.3MB 67.1MB/s 0.1s
New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.205 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/f/DATASETS/DataSED/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=128, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yoho_rgb_train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/mnt/f/DATASETS/DataSED/yolo_dataset/runs, quantize=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=detect, tile=False, time=None, tracker=tracktrack.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=4, workspace=None
Overriding model.yaml nc=80 with nc=22

                   from  n    params  module                                       arguments
  0                  -1  1       176  ultralytics.nn.modules.conv.Conv             [1, 16, 3, 2]
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]
  8                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]
  9                  -1  1    164608  ultralytics.nn.modules.block.SPPF            [256, 256, 5, 3, True]
 10                  -1  1    249728  ultralytics.nn.modules.block.C2PSA           [256, 256, 1]
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 13                  -1  1    119808  ultralytics.nn.modules.block.C3k2            [384, 128, 1, True]
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 16                  -1  1     34304  ultralytics.nn.modules.block.C3k2            [256, 64, 1, True]
 17                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]
 18            [-1, 13]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 19                  -1  1     95232  ultralytics.nn.modules.block.C3k2            [192, 128, 1, True]
 20                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]
 21            [-1, 10]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 22                  -1  1    463104  ultralytics.nn.modules.block.C3k2            [384, 256, 1, True, 0.5, True]
 23        [16, 19, 22]  1    249756  ultralytics.nn.modules.head.Detect           [22, 1, True, [64, 128, 256]]
YOLO26n summary: 260 layers, 2,512,092 parameters, 2,512,092 gradients, 5.9 GFLOPs

Remapped 1/22 cls head rows from pretrained weights by class name
Transferred 708/708 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
WARNING ⚠️ train: Slow image access detected (ping: 3.5±0.6 ms, read: 24.5±2.1 MB/s, size: 1348.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
train: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/train... 3005 i…: 100% ━━━━━━━━━━━━ 3005/3005 35.5it/s 1:25
train: New cache created: /mnt/f/DATASETS/DataSED/yolo_dataset/labels/train.cache
AutoBatch: Computing optimal batch size for imgsz=1024 at 60.0% GPU memory utilization.
AutoBatch: CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU) 7.96G total, 0.12G reserved, 6.82G available
      Params      GFLOPs GPU_mem (GiB)  forward (ms) backward (ms)                   input                  output
     2543804       15.91         0.429           310         948.6      (1, 1, 1024, 1024)                    list
     2543804       31.83         0.883         140.4         786.7      (2, 1, 1024, 1024)                    list
     2543804       63.66         1.868         161.6           668      (4, 1, 1024, 1024)                    list
     2543804       127.3         3.754         236.4         878.7      (8, 1, 1024, 1024)                    list
     2543804       254.6         7.512         231.9          1142     (16, 1, 1024, 1024)                    list
AutoBatch: Using batch-size 8 for CUDA:0 3.74G/6.82G (55%) ✅
train: Fast image access ✅ (ping: 2.4±0.7 ms, read: 194.5±33.4 MB/s, size: 1075.8 KB)
train: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/train.cache... …: 100% ━━━━━━━━━━━━ 3005/3005 1.4Git/s 0.0s
WARNING ⚠️ val: Slow image access detected (ping: 2.7±0.7 ms, read: 22.0±4.1 MB/s, size: 1102.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/val... 366 images,…: 100% ━━━━━━━━━━━━ 366/366 35.6it/s 10.3s
val: New cache created: /mnt/f/DATASETS/DataSED/yolo_dataset/labels/val.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and determining best 'optimizer' and 'lr0' automatically...
optimizer: AdamW(lr=0.000385, momentum=0.9) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0005), 126 bias(decay=0.0)
Plotting labels to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train/labels.jpg...
Using 3005 train, 366 val images for fraction=1.0 at imgsz=1024
Using 4 dataloader workers
Logging results to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train
Starting training for 128 epochs...

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      1/128      3.34G      1.937      40.22    0.04928         25       1024: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:26
                 Class     Images  Instances      Box(P          R      mAP50  …: 100% ━━━━━━━━━━━━ 23/23 1.8it/s 12.7s
                   all        366        832      0.463       0.16      0.135     0.0716

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      2/128      3.88G      1.213      4.739    0.02395         23       1024: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:26
                 Class     Images  Instances      Box(P          R      mAP50  m…: 100% ━━━━━━━━━━━━ 23/23 3.1it/s 7.5s
                   all        366        832      0.521       0.28      0.288      0.152

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/128      3.88G      1.131      3.317     0.0205         21       …: 82% ━━━━━━━━━╸── 309/376 2      3/128      3.88G      1.126      3.267    0.02033     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:24
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.404      0.374      0.368      0.216

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/128      3.88G      1.084      2.906    0.01934     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.534      0.413       0.42       0.28

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      5/128      3.88G      1.028      2.651    0.01742     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.477       0.47      0.473      0.314

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      6/128      3.88G     0.9863       2.52    0.01612     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.638      0.481      0.512      0.344

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      7/128      3.88G     0.9626      2.367    0.01545     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.616      0.501      0.514      0.342

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      8/128      3.88G     0.9392      2.269    0.01476     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.3s
                   all        366        832      0.512      0.504      0.528      0.372

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      9/128      3.88G     0.9224      2.226    0.01469     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.646      0.522      0.562      0.404

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     10/128      3.88G     0.8878      2.143    0.01378     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.659      0.514      0.568      0.393

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     11/128      3.88G      0.877      2.115     0.0133     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.645      0.569      0.585      0.424

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     12/128      3.88G     0.8745      2.062    0.01322     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.653      0.556       0.59       0.42

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     13/128      3.88G     0.8563      2.028    0.01258     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.655      0.554      0.591      0.421

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     14/128      3.88G     0.8377      1.974    0.01248     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.694      0.535      0.601      0.436

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     15/128      3.88G     0.8227      1.911    0.01209     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.0s
                   all        366        832       0.68      0.554      0.599      0.431

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     16/128      3.88G     0.8492      1.914    0.01239     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832       0.63      0.601      0.613       0.44

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     17/128      3.88G     0.8276      1.889    0.01209     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.662      0.594      0.636      0.465

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     18/128      3.88G     0.8265      1.853    0.01207     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.664       0.56      0.613      0.436

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     19/128      3.88G     0.8268      1.814    0.01211     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.712      0.593       0.63       0.46

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     20/128      3.88G     0.8059      1.814    0.01165     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.722      0.593       0.64       0.46

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     21/128      3.88G     0.8021      1.764    0.01157     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.689      0.637      0.656      0.479

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     22/128      3.88G      0.806      1.741    0.01146     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.724      0.594      0.664      0.488

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     23/128      3.88G     0.7981      1.721    0.01134     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.656      0.616      0.652      0.477

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     24/128      3.88G     0.7953      1.736    0.01132     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.687      0.613      0.659      0.471

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     25/128      3.88G     0.7931      1.724    0.01124     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.734      0.578      0.657      0.482

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     26/128      3.88G     0.7888      1.676    0.01121     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.714      0.601      0.659      0.489

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     27/128      3.88G      0.782      1.671    0.01104     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832       0.75      0.604      0.672      0.508

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     28/128      3.88G     0.7734      1.675    0.01078     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832        0.7      0.636      0.687      0.504

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     29/128      3.88G     0.7561       1.66    0.01057     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.768       0.63       0.69      0.513

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     30/128      3.88G     0.7612      1.633    0.01064     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.792      0.616      0.685      0.518

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     31/128      3.88G     0.7575      1.587    0.01046     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.762      0.613      0.685      0.513

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     32/128      3.88G       0.77      1.567    0.01066     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832       0.74      0.611      0.687      0.504

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     33/128      3.88G     0.7573      1.561     0.0106     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.768      0.615      0.686      0.514

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     34/128      3.88G      0.756      1.586    0.01061     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.703      0.627      0.677       0.51

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     35/128      3.88G      0.734      1.522    0.01007     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.747      0.615      0.691      0.528

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     36/128      3.88G     0.7391      1.548    0.01033     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.753      0.638      0.697       0.52

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     37/128      3.88G     0.7366      1.523   0.009928     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.779      0.641      0.704      0.523

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     38/128      3.88G     0.7286      1.506   0.009931     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.802      0.607      0.695      0.521

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     39/128      3.88G     0.7358      1.516   0.009996     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.798       0.62        0.7       0.52

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     40/128      3.88G     0.7239      1.487   0.009937     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.742      0.642      0.695      0.518

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     41/128      3.88G     0.7287      1.496    0.00963     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.743      0.645      0.695      0.529

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     42/128      3.88G     0.7161      1.474   0.009596     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.813      0.647      0.713      0.543

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     43/128      3.88G     0.7239      1.457   0.009579     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.758      0.649      0.708      0.542

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     44/128      3.88G     0.7244      1.487   0.009637     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.779      0.662      0.718      0.545

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     45/128      3.88G     0.7046      1.446   0.009512     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.787       0.65      0.712      0.539

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     46/128      3.88G     0.7119      1.453   0.009447     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.824      0.628      0.725      0.558

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     47/128      3.88G     0.7196       1.46   0.009487     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.779      0.635      0.716      0.551

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     48/128      3.88G     0.7068      1.402   0.009258     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.809      0.641      0.713      0.548

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     49/128      3.88G     0.6964      1.421   0.009182     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.763      0.656      0.712      0.543

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     50/128      3.88G     0.7038      1.405   0.009433     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.785      0.658      0.719      0.542

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     51/128      3.88G     0.6983      1.376   0.009536     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.804      0.667      0.726      0.551

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     52/128      3.88G     0.6936      1.375   0.009201     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.754      0.648      0.708      0.537

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     53/128      3.88G     0.6992      1.357   0.009264     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.776      0.638       0.71       0.54

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     54/128      3.88G     0.6933      1.386   0.009263     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832       0.81      0.631      0.714      0.537

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     55/128      3.88G     0.6848      1.341   0.008997     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.817      0.664       0.73      0.554

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     56/128      3.88G     0.6903      1.367   0.008842     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.786      0.659      0.716      0.546

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     57/128      3.88G     0.6868      1.332   0.008948     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.744      0.675      0.721      0.552

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     58/128      3.88G      0.684      1.321   0.008842     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.782      0.634      0.716      0.546

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     59/128      3.88G     0.6674      1.318   0.008808     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.737      0.658      0.717       0.55

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     60/128      3.88G     0.6801      1.341   0.008972     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.781      0.641      0.722      0.555

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     61/128      3.88G     0.6688      1.282    0.00873     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832        0.8      0.632      0.723      0.556

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     62/128      3.88G     0.6761      1.294   0.008703     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.804      0.639      0.717      0.553

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     63/128      3.88G     0.6748       1.32   0.008885     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.775      0.665      0.724       0.56

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     64/128      3.88G     0.6754      1.281   0.008586     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.771      0.668      0.718      0.553

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     65/128      3.88G     0.6838      1.311   0.008718     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.808      0.641      0.722      0.557

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     66/128      3.88G      0.667      1.287   0.008526     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.798      0.641      0.718      0.557

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     67/128      3.88G      0.655      1.281   0.008317     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.744      0.658      0.715      0.561

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     68/128      3.88G     0.6678      1.255   0.008665     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.769      0.673      0.721      0.564

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     69/128      3.88G     0.6481      1.257   0.008228     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.773      0.662      0.721      0.559

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     70/128      3.88G     0.6549      1.222   0.008266     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832       0.77      0.671      0.719      0.561

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     71/128      3.88G     0.6521       1.26   0.008334     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.779       0.66      0.726      0.569

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     72/128      3.88G     0.6578      1.245   0.008311     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.786      0.668      0.726      0.562

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     73/128      3.88G     0.6494      1.227   0.008358     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832       0.79      0.674      0.729      0.568

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     74/128      3.88G       0.64       1.19   0.008102     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.819      0.655      0.738      0.574

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     75/128      3.88G      0.647      1.225   0.008161     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.0s
                   all        366        832      0.803      0.658      0.726      0.572

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     76/128      3.88G     0.6395      1.201   0.007944     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.814      0.656      0.737      0.568

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     77/128      3.88G     0.6397      1.197   0.008142     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.811      0.666      0.739      0.575

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     78/128      3.88G     0.6446      1.239   0.008056     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.776      0.669      0.728      0.566

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     79/128      3.88G     0.6531      1.205   0.008056     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.751      0.672      0.728      0.571

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     80/128      3.88G      0.639       1.21   0.008102     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.773      0.679      0.737      0.573

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     81/128      3.88G     0.6285      1.208   0.007922     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.776      0.688       0.74      0.576

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     82/128      3.88G     0.6328      1.207    0.00811     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.0s
                   all        366        832      0.806      0.678       0.74      0.579

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     83/128      3.88G     0.6295      1.197   0.007841     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832       0.78      0.689      0.738      0.578

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     84/128      3.88G     0.6213      1.174   0.007602     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.822      0.647       0.73      0.574

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     85/128      3.88G     0.6378      1.177   0.007851     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.786       0.68      0.743       0.58

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     86/128      3.88G     0.6196      1.185   0.007857     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.803      0.681      0.738      0.576

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     87/128      3.88G     0.6296      1.192   0.007856     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.745      0.692      0.733      0.574

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     88/128      3.88G     0.6149      1.157   0.007654     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.766      0.693      0.736      0.574

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     89/128      3.88G      0.621      1.164   0.007677     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.793      0.679      0.736      0.582

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     90/128      3.88G     0.6121      1.153   0.007708     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.775      0.669      0.733       0.58

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     91/128      3.88G     0.6218      1.147   0.007637     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.796      0.658      0.734       0.57

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     92/128      3.88G      0.622      1.119   0.007717     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.793      0.663      0.731      0.576

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     93/128      3.88G     0.6255      1.134   0.007651     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.815      0.657      0.737      0.578

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     94/128      3.88G     0.6113      1.124   0.007412     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.804       0.67      0.741      0.583

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     95/128      3.88G     0.6259      1.158   0.007508     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.761      0.691      0.741      0.583

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     96/128      3.88G     0.6139      1.128   0.007486     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832       0.78      0.685      0.735      0.582

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     97/128      3.88G     0.6061      1.134   0.007334     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.782       0.69      0.738      0.585

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     98/128      3.88G     0.6194      1.138   0.007414     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.0s
                   all        366        832      0.828      0.649      0.734      0.584

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
     99/128      3.88G     0.5961      1.098   0.007281     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.817      0.669      0.736      0.581

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    100/128      3.88G      0.598      1.118   0.007344     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.808      0.679      0.737      0.586

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    101/128      3.88G     0.6111      1.114    0.00746     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.768      0.688      0.732      0.578

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    102/128      3.88G     0.6088      1.116   0.007299     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.802       0.69      0.734      0.584

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    103/128      3.88G     0.6005      1.124   0.007197     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832       0.79      0.681      0.735      0.586

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    104/128      3.88G     0.5989      1.101   0.007469     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.803      0.679      0.738      0.582

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    105/128      3.88G     0.6013      1.106   0.007307     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.792      0.675      0.737      0.584

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    106/128      3.88G     0.5913       1.08    0.00707     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.833      0.659      0.741      0.588

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    107/128      3.88G      0.598      1.078   0.007263     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.826      0.668      0.737      0.587

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    108/128      3.88G      0.604      1.109   0.007152     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.792      0.684      0.738      0.589

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    109/128      3.88G     0.5935        1.1   0.007035     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.796      0.682      0.738      0.587

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    110/128      3.88G     0.5807      1.079   0.006972     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.785      0.692      0.745       0.59

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    111/128      3.88G     0.5937      1.068   0.007061     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.799      0.698       0.75      0.594

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    112/128      3.88G     0.5918      1.073   0.007038     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.772      0.691      0.743      0.591

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    113/128      3.88G      0.596      1.091   0.006959     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.776      0.682      0.737      0.586

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    114/128      3.88G     0.5866      1.059      0.007     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.828       0.66      0.741      0.587

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    115/128      3.88G     0.5817      1.076   0.007019     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.811      0.669      0.744      0.588

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    116/128      3.88G     0.5848      1.075   0.006847     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:23
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.807      0.681      0.742      0.589

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    117/128      3.88G     0.5792      1.062   0.006983     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.801      0.678      0.743       0.59

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    118/128      3.88G     0.5774      1.058   0.006895     …: 100% ━━━━━━━━━━━━ 376/376 2.6it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.815      0.671      0.743       0.59
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    119/128      3.88G     0.6459     0.8545    0.01149     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:22
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.794      0.685      0.737      0.578

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    120/128      3.88G     0.6334     0.8029    0.01091     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.807      0.686       0.74      0.583

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    121/128      3.88G     0.6153     0.7768    0.01042     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.822       0.68      0.743      0.586

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    122/128      3.88G     0.6171     0.7692    0.01047     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.823      0.676       0.75      0.589

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    123/128      3.88G     0.6151      0.743    0.01042     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.1s
                   all        366        832      0.824      0.673      0.746       0.59

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    124/128      3.88G     0.6144     0.7644    0.01025     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.822      0.673      0.748      0.591

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    125/128      3.88G     0.6061     0.7429   0.009995     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.0s
                   all        366        832      0.808      0.682      0.748      0.591

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    126/128      3.88G     0.5991     0.7418    0.00982     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.2it/s 7.2s
                   all        366        832      0.798      0.687       0.75       0.59

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    127/128      3.88G     0.6061     0.7178   0.009916     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.798      0.684      0.746      0.591

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
    128/128      3.88G     0.5936     0.7181   0.009739     …: 100% ━━━━━━━━━━━━ 376/376 2.7it/s 2:21
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 3.3it/s 7.1s
                   all        366        832      0.792       0.69      0.748      0.591

128 epochs completed in 5.479 hours.
Optimizer stripped from /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train/weights/last.pt, 5.6MB
Optimizer stripped from /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train/weights/best.pt, 5.6MB

Validating /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train/weights/best.pt...
Ultralytics 8.4.205 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO26n summary (fused): 121 layers, 2,394,550 parameters, 0 gradients, 5.3 GFLOPs
                 Class     Images  Instances      Box(P       …: 100% ━━━━━━━━━━━━ 23/23 2.9it/s 8.0s
                   all        366        832        0.8      0.696       0.75      0.594
                 Bells         13         19      0.835      0.895      0.881      0.695
                 Birds         26         33      0.668      0.333      0.401      0.295
   Cat fight and moans          7         26      0.843      0.885      0.936      0.753
          Chicken coop         13         28      0.915      0.786      0.867      0.598
  Cicadas and crickets         30         71       0.92      0.831      0.924      0.785
Crows seagulls and magpies         12         16      0.633      0.312      0.451      0.303
Dog barkings and howlings         21         43       0.77      0.544      0.639      0.462
        Glass breaking          5         14      0.932      0.786      0.915      0.693
                  Horn          9         19      0.552      0.421      0.416      0.309
         Jet aircrafts         28         32      0.731      0.595      0.587        0.4
Lawn mower brush cutter and olive shaker         25         36      0.886      0.778      0.851      0.697
                 Music         28         41      0.932      0.854      0.872      0.797
   Propeller aircrafts         37         56      0.782      0.839      0.859      0.723
     Sirens and alarms         19         25      0.877       0.84      0.919      0.772
Thunder fireworks and gunshot         15         31      0.805      0.797      0.787      0.539
                 Train         38         50      0.882       0.72      0.821       0.64
Vacuum cleaner fan and hairdryer         21         30      0.965      0.919      0.958      0.882
        Vehicle idling         24         31      0.345       0.29      0.205      0.116
       Vehicle pass-by         48         61      0.694      0.557      0.633       0.41
                Voices         34         49      0.833      0.571      0.705      0.494
          Wind turbine         29         31      0.932       0.88      0.936      0.852
              Workshop         45         90      0.861      0.889      0.931       0.85
Speed: 0.2ms preprocess, 17.6ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train

Entrenamiento finalizado.
Resultados en: /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train`

`(aexxx) aexxtool@AExxTool:~/YOHO-by-AEx/src$ python3 evaluate.py
============================================================
EVALUACION YOLO26n - DATASET RGB (TEST)
============================================================
Modelo: /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_train/weights/best.pt
Existe: True
Data:   /mnt/f/DATASETS/DataSED/yolo_dataset/data.yaml
Existe: True
Ultralytics 8.4.205 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO26n summary (fused): 121 layers, 2,394,550 parameters, 0 gradients, 5.3 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 2.7±0.6 ms, read: 21.7±4.8 MB/s, size: 1003.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/…: 100% ━━━━━━━━━━━━ 366/366 35.6it/s 10.3s
val: New cache created: /mnt/f/DATASETS/DataSED/yolo_dataset/labels/test.cache
                 Class     Images  Instances      Box(P      …: 100% ━━━━━━━━━━━━ 46/46 4.1it/s 11.2s
                   all        366        910      0.805      0.707      0.763      0.587
                 Bells         13         29      0.923      0.897      0.951      0.745
                 Birds         25         37      0.557      0.459      0.403      0.255
   Cat fight and moans          6         23      0.827      0.833      0.905       0.73
          Chicken coop         11         25       0.88      0.877      0.863      0.616
  Cicadas and crickets         31         83      0.905      0.735       0.88      0.679
Crows seagulls and magpies         14         24       0.67        0.5      0.569      0.494
Dog barkings and howlings         13         36      0.688      0.639      0.636       0.35
        Glass breaking          8         26      0.858      0.808        0.9      0.618
                  Horn         11         18      0.613      0.389       0.41      0.207
         Jet aircrafts         23         24      0.633      0.417      0.473      0.348
Lawn mower brush cutter and olive shaker         24         33      0.879       0.88      0.889      0.725
                 Music         28         40      0.894      0.843       0.93      0.836
   Propeller aircrafts         38         68      0.755      0.726      0.793      0.651
     Sirens and alarms         24         38      0.938      0.947      0.972      0.854
Thunder fireworks and gunshot         13         28      0.898      0.631      0.794      0.463
                 Train         36         45      0.919      0.754      0.864      0.733
Vacuum cleaner fan and hairdryer         19         30      0.907        0.9      0.947      0.904
        Vehicle idling         28         35       0.53      0.419      0.458      0.296
       Vehicle pass-by         46         68      0.824      0.549      0.677      0.401
                Voices         42         63      0.695      0.578      0.632      0.389
          Wind turbine         37         45      0.923      0.867      0.892      0.755
              Workshop         46         92      0.988       0.91      0.945      0.864
Speed: 0.5ms preprocess, 24.9ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_rgb_test

Metricas sobre TEST (RGB, YOLO26n):
  mAP@0.5:      0.7629
  mAP@0.5:0.95: 0.5868
  Precision:    0.8047
  Recall:       0.7071`

In [ ]:
from PIL import Image
from pathlib import Path

img_dir = Path("/mnt/f/DATASETS/DataSED/yolo_dataset/images/train")
imgs = sorted(img_dir.glob("*.png"))[:100]
for p in imgs:
    img = Image.open(p)
    print(f"{p.name}: mode={img.mode}")